In [ ]:
Colab 코드
# [실습 1] retail_sales_reg1.csv : 다중회귀 01

from google.colab import drive
from pathlib import Path
import pandas as pd
import statsmodels.formula.api as smf

# 1. 드라이브 마운트
# drive.mount('/content/drive')

# 2. 경로 설정
DATA_DIR = Path('/content/type3')
file_path = DATA_DIR / 'retail_sales_reg1.csv'

# 3. 데이터 불러오기
df = pd.read_csv(file_path)

print('[데이터 상위 5행]')
print(df.head(), '\n')

print('[기본 정보]')
print(df.info(), '\n')

# 4. 회귀모형 적합, Ordinary Least Squares(최소자승법)
model = smf.ols('sales ~ ad_cost + staff + event_cnt', data=df).fit()

# 5. 결과 요약
print('[회귀 요약 결과]')
print(model.summary())

# 6. 핵심 결과 추출
print('\n[회귀계수]')
print(model.params)

print('\n[p-value]')
print(model.pvalues)

print('\n[R-squared]')
print(model.rsquared)

# 7. 광고비 10단위 증가 효과
beta_ad = model.params['ad_cost']
print('\n광고비 10단위 증가 시 예상 매출 변화:', beta_ad * 10)

# 8. 유의한 변수 목록
sig_vars = model.pvalues[model.pvalues < 0.05].index.tolist()
print('\n유의한 변수:', sig_vars)



핵심 설명

샘플 CSV 기준 실행 결과는 다음처럼 해석할 수 있습니다.
• 회귀식:[    sales = 57.8033 + 2.7975(ad_cost) + 4.2634(staff) + 20.4831(event_cnt)]
• ad_cost계수: 2.7975
• staff계수: 4.2634
• event_cnt계수: 20.4831
• 세 변수의 p-value가 모두 0.05보다 작음
• R² = 0.8858
• 광고비 10단위 증가 시 예상 매출 변화 = 27.9746

즉, 이 모형에서는 광고비, 직원 수, 행사 횟수가 모두 월매출에 유의한 영향을 주며, 특히 행사 횟수의 효과 크기가 크게 나타납니다. 또 R²가 0.8858이므로, 이 회귀식은 매출 변동의 약 88.58%를 설명한다고 해석할 수 있습니다.


해석 문장
다중회귀분석 결과, 광고비(ad_cost), 직원 수(staff), 주말행사 횟수(event_cnt)는 모두 월매출(sales)에 통계적으로 유의한 영향을 미치는 변수로 나타났다. 광고비의 회귀계수는 2.7975로, 다른 조건이 동일할 때 광고비가 1단위 증가하면 매출은 평균적으로 약 2.7975증가한다고 해석할 수 있다. 결정계수는 R²=0.8858로, 이 모형은 월매출 변동의 약 88.58%를 설명한다.


실습 2) weather_reg2.csv
다중회귀 02: 예측값·신뢰구간·오차지표
문제 상황
환경 데이터 분석팀은 기온(temperature)을 설명하기 위해 일사량(solar), 바람(wind), 오존(o3)을 활용한 다중회귀모형을 만들었다.

강의 목표
• 회귀계수 해석
• p-value로 변수 유의성 확인
• 특정 조건에서의 예측값 계산
• 신뢰구간 확인
• MSE/RMSE 해석


Colab 코드
# [실습 2] weather_reg2.csv : 다중회귀 02

from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.metrics import mean_squared_error

# 1. 드라이브 마운트
#drive.mount('/content/drive')

# 2. 경로 설정
DATA_DIR = Path('/content/type3)
file_path = DATA_DIR / 'weather_reg2.csv'

# 3. 데이터 불러오기
df = pd.read_csv(file_path)

print('[데이터 상위 5행]')
print(df.head(), '\n')

print('[기본 정보]')
print(df.info(), '\n')

# 4. 회귀모형 적합
model = smf.ols('temperature ~ solar + wind + o3', data=df).fit()

# 5. 결과 요약
print('[회귀 요약 결과]')
print(model.summary())

# 6. 핵심 계수와 p-value
print('\n[회귀계수]')
print(model.params)

print('\n[p-value]')
print(model.pvalues)

print('\n[R-squared]')
print(model.rsquared)

# 7. 특정 조건 예측
new_df = pd.DataFrame({
    'solar': [100],
    'wind': [5],
    'o3': [30]
})

pred = model.predict(new_df)
print('\n[예측값]')
print(pred)

# 8. 신뢰구간 / 예측 요약표
pred_frame = model.get_prediction(new_df).summary_frame(alpha=0.05)
print('\n[예측 요약표]')
print(pred_frame)

# 9. 오차지표 계산
y_true = df['temperature']
y_pred = model.predict(df)

mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)

print('\n[MSE / RMSE]')
print('MSE :', mse)
print('RMSE:', rmse)



핵심설명
샘플 CSV 기준 실행 결과는 다음처럼 정리할 수 있습니다.
• 회귀식:[       temperature = 11.7807 + 0.0439(solar) - 0.9220(wind) + 0.2437(o3)]
• solar계수: 0.0439
• wind계수: -0.9220
• o3계수: 0.2437
• 세 변수의 p-value가 모두 0.05보다 작음
• R² = 0.8816
• solar=100, wind=5, o3=30일 때 예측값: 18.8757
• MSE = 4.9019
• RMSE = 2.2140

즉, 일사량과 오존은 기온을 높이는 방향, 바람은 기온을 낮추는 방향으로 유의하게 작용합니다. 예측 온도는 약 18.88이며, RMSE가 약 2.21이라는 것은 모델 예측이 실제값과 평균적으로 약 2.21 정도 차이 난다는 뜻입니다.


해석 문장
다중회귀분석 결과, solar, wind, o3는 모두 기온(temperature)에 유의한 영향을 주는 변수로 나타났다. wind의 회귀계수는 -0.9220이므로, 다른 조건이 같을 때 바람이 1단위 증가하면 기온은 평균적으로 약 0.9220감소한다고 해석할 수 있다. 또한 solar=100, wind=5, o3=30인 조건에서 예측된 기온은 18.8757이며, RMSE는 2.2140으로 예측 오차의 평균 크기가 약 2.21 수준임을 의미한다.